## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

## Setting up SendGrid


Please visit Sendgrid at: https://sendgrid.com/

(Sendgrid is a Twilio company for sending emails.)

__If SendGrid gives you problems, see the alternative implementations on Q29 of my FAQ page at https://edwarddonner.com/faq that includes "Resend Email" in community_contributions/2_lab2_with_resend_email and just skipping email altogether.__

Setting up a SendGrid account is free! (at least, for me, right now).

Once you've created an account, click on:

Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:

`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:

Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.

In [7]:
from dotenv import load_dotenv
import os
from agents import Agent, Runner, trace, function_tool
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio

In [3]:
load_dotenv(override=True)

True

In [4]:
# Let's just check emails are working for you

def send_test_email():
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tangotew@gmail.com")  # Change to your verified sender
    to_email = To("tangogatdet76@gmail.com")  # Change to your recipient
    content = Content("text/plain", "This is an important test email")
    mail = Mail(from_email, to_email, "Test email", content).get()
    response = sg.client.mail.send.post(request_body=mail)
    print(response.status_code)

send_test_email()

202


### Did you receive the test email

If you get a 202, then you're good to go!

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard

Or use the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow
Lets create three different agents, well doing the same thing but slightly differently, and see how they can work together.

In [5]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [6]:
sale_agent1 = Agent(
  name="Professional Pete",
  instructions=instructions1,
  model="gpt-4o-mini"
)

sale_agent2 = Agent(
  name="Humorous Hannah",
  instructions=instructions2,
  model="gpt-4o-mini"
)

sale_agent3 = Agent(
  name="Busy Bob",
  instructions=instructions3,
  model="gpt-4o-mini"
)

In [11]:
# We're going to run the agent using the stream
result = Runner.run_streamed(
  sale_agent1,
  input="Write a cold email to a potential customer who might be interested in our product, ComplAI."
) # this stream returns a coroutine, so we need to await it to get the result

async for event in result.stream_events():
  if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
    print(event.data.delta, end="", flush=True)


Subject: Simplify Your SOC2 Compliance Process with ComplAI

Hi [Recipient's Name],

I hope this email finds you well.

Navigating the complexities of SOC2 compliance can be a daunting task, often leading to significant resource expenditure and stress. At ComplAI, we understand the challenges faced by organizations like yours, which is why we’ve developed a powerful SaaS tool designed to streamline the compliance process and prepare you for audits with ease.

Our AI-driven platform offers:

- **Automated Compliance Tracking:** Stay informed and proactive with real-time insights into your compliance status.
- **Seamless Documentation Management:** Easily create, manage, and share documentation, reducing your administrative burden.
- **Audit Readiness:** Get ready for audits with confidence, knowing that our tool highlights any gaps and provides actionable recommendations.

Many organizations have already transformed their compliance processes with ComplAI, saving valuable time and resou

In [12]:
# run them all using asyncio.gather to run them concurrently
message = "write a cold email."

with trace("Parallel cold emails"):
  results = await asyncio.gather(
    Runner.run(sale_agent1, input=message),
    Runner.run(sale_agent2, input=message),
    Runner.run(sale_agent3, input=message)
  )
  
outputs = [result.final_output for result in results]

for output in outputs:
  print("\n\n---\n\n")
  print(output)



---


Subject: Simplifying Your SOC 2 Compliance Journey

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m with ComplAI. We specialize in helping companies like yours navigate the complexities of SOC 2 compliance with our AI-powered SaaS tool.

In today’s landscape, maintaining compliance is crucial, not just for audits but also for building trust with your clients. Our platform streamlines the entire process—from documentation to monitoring—saving you valuable time and resources while ensuring that you meet all necessary requirements.

I’d love to schedule a brief call to discuss how ComplAI can support your compliance efforts and help you prepare for upcoming audits seamlessly. Would you be available for a quick chat this week?

Looking forward to your response.

Best regards,

[Your Name]  
[Your Position]  
ComplAI  
[Your Contact Information]  
[Your LinkedIn Profile or Company Website]  


---


Subject: 🌟 Unlock the Secret to Stress-F

In [15]:
# now lets use Evaluation optimizer design pattern to evaluate which email is best and why, and then rewrite the worst performing one to be better.
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model="gpt-4o-mini"
)



In [16]:
# now we orchestrate the evaluation and rewriting of the worst performing email
with trace("write a cold email."):
  results = await asyncio.gather(
    Runner.run(sale_agent1, input=message),
    Runner.run(sale_agent2, input=message),
    Runner.run(sale_agent3, input=message)
  )
outputs = [result.final_output for result in results]

# add evaluator agent to pick the best email
emails = "Cold sale emails: \n\n" + "\n\nEmail:\n\n".join(outputs)

best_email = await Runner.run(sales_picker, input=emails)
print("Best email:\n\n", best_email.final_output)

Best email:

 Subject: Ready to Make SOC 2 Compliance Less Painful? 🎉

Hi [Recipient's Name],

Hope you’re surviving your inbox jungle! 🌴 I wanted to drop you a quick note about something that might just make your day a little brighter: SOC 2 compliance.

Ah, the sweet sounds of audits! They can feel like dentist visits—essential but oh-so-dreadful. What if I told you that with ComplAI, you could turn this compliance headache into a walk in the park? 🚶‍♂️🌳

Our SaaS tool is like your friendly neighborhood superhero for compliance—minus the cape but packed with AI-powered insights! It streamlines your processes, keeps everything organized, and helps you sail through audits smoother than a buttered dolphin. 🐬🧈

Let’s chat about how we can help you conquer compliance without losing your sanity. When’s a good time for you? 

Looking forward to making your day (and audits) a whole lot easier!

Best,  
[Your Name]  
[Your Position]  
ComplAI  
[Your Contact Information]  
  
P.S. No capes re

Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [17]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model="gpt-4o-mini",
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model="gpt-4o-mini",
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model="gpt-4o-mini",
)

In [18]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model='gpt-4o-mini', model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [20]:
# simply wrap sendgrid function with a decorator to make it agentic tool

@function_tool
def send_email(body: str):
  """ Send an email with the given body using SendGrid API to all sale prospects """
  sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
  from_email = Email("tangotew@gmail.com")  
  to_email = To("tangogatdet76@gmail.com")
  content = Content("text/plain", body)
  mail = Mail(from_email, to_email, "Cold email from ComplAI", content)
  sg.client.mail.send.post(request_body=mail.get())
  return {"status": "sent successfully"}

In [21]:
send_email 

FunctionTool(name='send_email', description='Send an email with the given body using SendGrid API to all sale prospects', params_json_schema={'properties': {'body': {'title': 'Body', 'type': 'string'}}, 'required': ['body'], 'title': 'send_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x108a94f40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [22]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x114bca8e0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [27]:
description = "write a cold sale email" 
tools = []
for agent in [sale_agent1, sale_agent2, sale_agent3]: # loop through the agents and turn them into tools
  tool = agent.as_tool(tool_name=agent.name, tool_description=description)
  tools.append(tool)
tools.append(send_email)

## And now it's time for our Sales Manager - our planning agent

In [32]:
description = "Write a cold sales email"

#use for loop to turn each agent into a tool and add it to the tools list
tools = []
for agent in [sales_agent1, sales_agent2, sales_agent3]:
    tool = agent.as_tool(tool_name=agent.name, tool_description=description)
    tools.append(tool)
tools.append(send_email)

In [33]:
tools

[FunctionTool(name='Professional Sales Agent', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'Professional Sales Agent_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x110c65760>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='Engaging Sales Agent', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'Engaging Sales Agent_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x114bc8ea0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(

In [ ]:
instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""



In [43]:

message = "Send a cold sale email adress to  'Dear CEO Tango'"
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_email]
sale_manager = Agent(
  name="Sale Manager",
  instructions=instructions,
  model="gpt-4o-mini",
  tools=tools
)
with trace("Sales Manager"):
  result = await Runner.run(sale_manager, input=message)

### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across

In [44]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model="gpt-4o-mini")
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model="gpt-4o-mini")
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")

In [45]:
@function_tool
def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
    """ Send out an email with the given subject and HTML body to all sales prospects """
    sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
    from_email = Email("tangotew@gmail.com")  # Change to your verified sender
    to_email = To("tangogatdet76@gmail.com")  # Change to your recipient
    content = Content("text/html", html_body)
    mail = Mail(from_email, to_email, subject, content).get()
    sg.client.mail.send.post(request_body=mail)
    return {"status": "success"}

In [46]:
tools = [subject_tool, html_tool, send_html_email]

In [47]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."

emailer_agent = Agent(
    name="Emailer Agent",
    instructions=instructions,
    model="gpt-4o-mini",
    tools=tools,
    handoff_description="Convert an email to HTML and send it."
)

In [49]:
handoffs = [emailer_agent]

### Now we have 3 tools and 1 handoff

In [51]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""

sales_manager = Agent(
  name="Sales Manager",
  instructions=sales_manager_instructions,
  model="gpt-4o-mini",
  tools=tools,
  handoffs=handoffs
)

message = "Send a cold sale email adress to  'Dear CEO'"

with trace("Automated SDR"):
  result = await Runner.run(sales_manager, input=message)

[non-fatal] Tracing: server error 504, retrying.


### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

### This is the 2026 definition of an Agentic Framework in action:
- where we define agent to be an LLM that use tools in a loop to achieve its goal. compared to anthropic's definition;
- where they define an agent as a system where LLM controls the workflows, vs workflow definition where LLM is orchestrated via predefined code to achieve something.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

- research send grid to have an agent that can facilitate email sending service and replies.
- if the sender to respond to is mentioned, there should be email reader agent(wrapped as the tool)
  - this agent would read and pass data along but it should summarize any other important action and send me a push notification using push tool as well,
- once the content is back to the Main handler agent, then those would be hand offs to the reply agent.